In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shree1992/housedata/output.csv
/kaggle/input/datasets/shree1992/housedata/data.csv
/kaggle/input/datasets/shree1992/housedata/data.dat
/kaggle/input/datasets/zerotrace11/tech-company-dataset/tech_company_employee_data_1000.csv
/kaggle/input/datasets/lovishbansal123/titanic-dataset/Titanic-Dataset.csv
/kaggle/input/datasets/sagnikpatra/uci-adult-census-data-dataset/adult_train.csv
/kaggle/input/datasets/sagnikpatra/uci-adult-census-data-dataset/adult_test.csv
/kaggle/input/datasets/iammustafatz/diabetes-prediction-dataset/diabetes_prediction_dataset.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
uci = pd.read_csv('/kaggle/input/datasets/sagnikpatra/uci-adult-census-data-dataset/adult_train.csv')
diabetes = pd.read_csv('/kaggle/input/datasets/iammustafatz/diabetes-prediction-dataset/diabetes_prediction_dataset.csv')
titanic = pd.read_csv('/kaggle/input/datasets/lovishbansal123/titanic-dataset/Titanic-Dataset.csv')
house = pd.read_csv('/kaggle/input/datasets/shree1992/housedata/data.csv')

# impute missing values of occupation column of uci dataset

In [4]:
replace = uci['Occupation'].mode()[0]
uci['Occupation'] = uci['Occupation'].fillna(replace)
uci['Occupation'].isnull().sum()

np.int64(0)

# Outliers

# 1. find and mitigate(trimming and capping both) outliers on age column of diabetes dataset

In [5]:
diabetes['age'].skew()


np.float64(-0.05197899678256747)

In [6]:
diabetes['age'].describe()

count    100000.000000
mean         41.885856
std          22.516840
min           0.080000
25%          24.000000
50%          43.000000
75%          60.000000
max          80.000000
Name: age, dtype: float64

In [7]:
highest = diabetes['age'].mean() + 3*diabetes['age'].std()
highest                                                         

np.float64(109.43637561484539)

# 2. find and mitigate(trimming and capping both) outliers on price column of house dataset

In [8]:
Q1 = house['price'].quantile(0.25)
Q3 = house['price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [9]:
print(lower_bound)
print(upper_bound)

-175256.25
1153093.75


In [10]:
house_trimmed = house[(house['price'] >= lower_bound) & (house['price'] <= upper_bound)]
print(house_trimmed)

                     date          price  bedrooms  bathrooms  sqft_living  \
0     2014-05-02 00:00:00  313000.000000       3.0       1.50         1340   
2     2014-05-02 00:00:00  342000.000000       3.0       2.00         1930   
3     2014-05-02 00:00:00  420000.000000       3.0       2.25         2000   
4     2014-05-02 00:00:00  550000.000000       4.0       2.50         1940   
5     2014-05-02 00:00:00  490000.000000       2.0       1.00          880   
...                   ...            ...       ...        ...          ...   
4595  2014-07-09 00:00:00  308166.666667       3.0       1.75         1510   
4596  2014-07-09 00:00:00  534333.333333       3.0       2.50         1460   
4597  2014-07-09 00:00:00  416904.166667       3.0       2.50         3010   
4598  2014-07-10 00:00:00  203400.000000       4.0       2.00         2090   
4599  2014-07-10 00:00:00  220600.000000       3.0       2.50         1490   

      sqft_lot  floors  waterfront  view  condition  sqft_above

In [11]:
house_capped = house.copy()

house_capped['price'] = np.where(
    house_capped['price'] > upper_bound, 
    upper_bound, 
    np.where(house_capped['price'] < lower_bound, lower_bound, house_capped['price'])
)

In [12]:
print(f"Capped Max Price:   ${house_capped['price'].max():,.2f}")
print(f"Row count remains unchanged: {house_capped.shape[0]}")

Capped Max Price:   $1,153,093.75
Row count remains unchanged: 4600
